In [1]:
# Importando as bibliotecas
import time
inicio = time.time()

from sahi.utils.yolov8 import download_yolov8s_model
from sahi import AutoDetectionModel
from sahi.utils.cv import read_image
from sahi.utils.file import download_from_url
from sahi.predict import get_prediction, get_sliced_prediction, predict
from pathlib import Path
from IPython.display import Image
import pandas as pd
import os, sys
from scipy.io import netcdf
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import xarray as xr
import imantics

import warnings
warnings.filterwarnings("ignore")
%matplotlib inline


08/20/2024 09:12:40 - INFO - numexpr.utils -   NumExpr defaulting to 8 threads.


In [2]:
# Download YOLOv8 model
yolov8_model_path = "C:/Users/camil/Documents/Mestrado/projetos/EMB nao Colab/sahi/teste/best_v12.pt"
download_yolov8s_model(yolov8_model_path)


detection_model = AutoDetectionModel.from_pretrained(
    model_type='yolov8',
    model_path=yolov8_model_path,
    confidence_threshold=0.6,
    device="cpu" # or 'cuda:0'
)

In [3]:
# selecionando o arquivo a ser inferido
files = os.listdir()
filtered_files_png = [file_  for file_ in files if file_.startswith('S1A_IW_GRDH') & file_.endswith('.png')]
print('O arquivo selecionado foi {}.'.format(filtered_files_png[-1]))


O arquivo selecionado foi S1A_IW_GRDH_1SDV_20240603T080345_20240603T080410_054156_0695FE_2C90_Spk_Intensity_VV.png.


In [4]:
#result = get_sliced_prediction('S1A_IW_GRDH_1SDV_20240314T222728_20240314T222753_052984_066A03_7112_Spk_Intensity_VV.jpg',
result = get_sliced_prediction(filtered_files_png[-1],
detection_model,
slice_height=1280,
slice_width=1280,
overlap_height_ratio=0.1,
overlap_width_ratio=0.1)

result.export_visuals(export_dir="teste/")
Image("teste/prediction_visual.png")

# Access the object prediction list
object_prediction_list = result.object_prediction_list

# Convert to COCO annotation, COCO prediction, imantics, and fiftyone formats
#result.to_coco_annotations()
resul = result.to_coco_predictions(image_id=1)
#result.to_imantics_annotations()[:]
#result.to_fiftyone_detections()[:3]

n_emb = len((result.object_prediction_list))
print('Número de embarcações encontradas na imagem com sahi foi:', n_emb)

fim = time.time()
print(fim - inicio)

Performing prediction on 345 number of slices.
Número de embarcações encontradas na imagem com sahi foi: 6
390.79642367362976


In [5]:
# criando os dataframes para processamento dos resultados

df = pd.DataFrame.from_records(resul)
df2 = pd.DataFrame(df['bbox'].to_list(), columns=['x','y', 'width','height'])

#df[['x', 'y', 'width', 'height']] = df['bbox'].str.split(', ', True)
extracted_col = df["score"]
df2.insert(4, "score", extracted_col)

#criando os valores de pixel para lat e lon

df2['lon_pixel'] = df2['x'] + (df2['width']/2)
df2['lat_pixel'] = df2['y'] + (df2['height']/2)

# arredondando 'lat lon pixels'
df2['lat_pixel'] = df2['lat_pixel'].round(0).astype(int)
df2['lon_pixel'] = df2['lon_pixel'].round(0).astype(int)

# salvando a planilha com os alvos identificados pelo yolo
df2.to_csv('ship_yolo.csv', index=False, sep='\t') 


files = os.listdir()
filtered_files_nc = [file_  for file_ in files if file_.startswith('S1A_IW_GRDH') & file_.endswith('.nc') ]
print('O arquivo selecionado foi {}.'.format(filtered_files_nc[-1]))

ds = xr.open_dataset(filtered_files_nc[-1])


O arquivo selecionado foi S1A_IW_GRDH_1SDV_20240603T080345_20240603T080410_054156_0695FE_2C90_Spk.nc.


In [6]:
# obtendo os vertices da imagem NETCDF para plotagame no mapa
latitudes = ds['latitude']
longitudes = ds['longitude']
lat_vertice_superior_esquerdo = latitudes[0, 0].values
long_vertice_superior_esquerdo = longitudes[0, 0].values

lat_vertice_superior_direito = latitudes[0, -1].values
long_vertice_superior_direito = longitudes[0, -1].values

lat_vertice_inferior_direito = latitudes[-1, -1].values
long_vertice_inferior_direito = longitudes[-1, -1].values

lat_vertice_inferior_esquerdo = latitudes[-1, 0].values
long_vertice_inferior_esquerdo = longitudes[-1, 0].values

In [7]:
#lendo os arquivos

df=pd.read_csv('ship_yolo.csv', sep='\t')
df = pd.DataFrame(df)

a = list(np.arange(len(df.x)))
df.insert(7, "Latitude", a, True)
df.insert(8, "Longitude", a, True)
df.insert(9, "Label", a, True)

#obtendo os valores de LAT/LON de cada alvo do YOLO com base no arquivo NETCDF

for i in range(len(df.lon_pixel)):
    df['Latitude'][i]= ds.latitude.isel(y=df.lat_pixel[i], x=df.lon_pixel[i]).values
    df['Longitude'][i]= ds.longitude.isel(y=df.lat_pixel[i], x=df.lon_pixel[i]).values
    df['Label'][i]= i+1

df.to_csv('yolo_georef.csv', index=False, columns=['Label','Latitude','Longitude'], sep='\t')  
da=pd.read_csv('seavision.csv', sep='\t')
da = pd.DataFrame(da)
da

,Latitude,Longitude,Ship Type,Speed,Heading,Length,Beam,Age,Vessel Name
0,-17.70,-37.89,Unknown,4.6,253,10,5,"0h, 14m, 18s ago",Julianne I Boia Ii
1,-12.75,-38.63,Tanker,0.0,139,183,32,"0h, 05m, 09s ago",Dalmacija
2,-11.33,-29.72,Fishing,7.5,42,40,8,"0h, 15m, 47s ago",Escualo Cuatro
3,-12.97,-38.51,Cargo,0.0,225,180,30,"0h, 13m, 07s ago",Erhan
4,-15.37,-36.73,Cargo,10.1,190,293,40,"0h, 16m, 12s ago",Apl Turkey
...,...,...,...,...,...,...,...,...,...
155,-12.69,-38.64,Tug,0.0,47,26,8,"0h, 21m, 11s ago",Cl Cacao
156,-16.79,-38.70,Unknown,5.2,210,0,0,"0h, 17m, 42s ago",Amz Ii - Boia 6
157,-16.28,-38.14,Unknown,1.4,62,65,0,"0h, 17m, 50s ago",Ksk-boia2
158,-16.31,-38.28,Unknown,1.9,137,59,0,"0h, 18m, 58s ago",Anisio Pai I - Boia4


In [8]:
# Função para extrair os minutos
def extract_minutes(time_str):
    parts = time_str.split(', ')
    hours = int(parts[0].split('h')[0])
    minutes = int(parts[1].split('m')[0])
    return hours * 60 + minutes

# Criando uma coluna com a Idade do AIS em minutos e em fraçao de hora
da['Minutos_Tempo'] = da['Age'].apply(extract_minutes)

da['diff tempo']= 65-da['Minutos_Tempo']
da['hora_Tempo_diff'] = da['diff tempo']/60
da['Raio_alcance']=(da['Speed']*da['hora_Tempo_diff']).round(1)
da['Raio_alcance']= da['Raio_alcance']*1.2
da

,Latitude,Longitude,Ship Type,Speed,Heading,Length,Beam,Age,Vessel Name,Minutos_Tempo,diff tempo,hora_Tempo_diff,Raio_alcance
0,-17.70,-37.89,Unknown,4.6,253,10,5,"0h, 14m, 18s ago",Julianne I Boia Ii,14,51,0.850000,4.68
1,-12.75,-38.63,Tanker,0.0,139,183,32,"0h, 05m, 09s ago",Dalmacija,5,60,1.000000,0.00
2,-11.33,-29.72,Fishing,7.5,42,40,8,"0h, 15m, 47s ago",Escualo Cuatro,15,50,0.833333,7.44
3,-12.97,-38.51,Cargo,0.0,225,180,30,"0h, 13m, 07s ago",Erhan,13,52,0.866667,0.00
4,-15.37,-36.73,Cargo,10.1,190,293,40,"0h, 16m, 12s ago",Apl Turkey,16,49,0.816667,9.84
...,...,...,...,...,...,...,...,...,...,...,...,...,...
155,-12.69,-38.64,Tug,0.0,47,26,8,"0h, 21m, 11s ago",Cl Cacao,21,44,0.733333,0.00
156,-16.79,-38.70,Unknown,5.2,210,0,0,"0h, 17m, 42s ago",Amz Ii - Boia 6,17,48,0.800000,5.04
157,-16.28,-38.14,Unknown,1.4,62,65,0,"0h, 17m, 50s ago",Ksk-boia2,17,48,0.800000,1.32
158,-16.31,-38.28,Unknown,1.9,137,59,0,"0h, 18m, 58s ago",Anisio Pai I - Boia4,18,47,0.783333,1.80


In [9]:
# Criando um raio de incerteza do AIS

# Define o intervalo de erro (0.5 graus) == raio 50km
intervalo_de_erro = 0.25


# Cria um terceiro dataframe com os dados que possuem a mesma latitude e longitude
df2 = pd.DataFrame(columns=['Latitude', 'Longitude', 'Vessel Name', 'x', 'y', 'width', 'height','score', 'lon_pixel', 'lat_pixel', 'Label'])

# Função para verificar se a diferença entre as coordenadas está dentro do intervalo de erro
def dentro_do_intervalo(coord1, coord2):
    lat_diff = abs(coord1['Latitude'] - coord2['Latitude'])
    lon_diff = abs(coord1['Longitude'] - coord2['Longitude'])
    return lat_diff <= intervalo_de_erro and lon_diff <= intervalo_de_erro

# Adiciona ao Dataframe somente as linhas que possuem correlação +- 0.5°
for index1, row1 in df.iterrows():
    for index2, row2 in da.iterrows():
        if dentro_do_intervalo(row1, row2):
            df2 = df2.append(row1, ignore_index=True)
            df2.at[df2.index[-1], 'Vessel Name'] = row2['Vessel Name']  # Adicione a coluna "Vessel Name" do da


# Salva o dataframe em um arquivo csv
df2.to_csv('both_duplicated.csv', index=False, sep='\t') 

df3 = df2.drop_duplicates(subset=['Label']) #reduzindo o DF para ter apenas 1 imagem por EMB identificada no YOLO.



In [10]:
df3

,Latitude,Longitude,Vessel Name,x,y,width,height,score,lon_pixel,lat_pixel,Label
0,-13.581248,-36.787758,Amity,8026.569092,13071.831421,38.123291,53.208862,0.828042,8046.0,13098.0,1.0
1,-13.378147,-36.938988,Mai Tai,10128.824280,11281.214966,30.093994,56.912964,0.827811,10144.0,11310.0,2.0
2,-13.686136,-36.606274,Amity,5853.113525,13726.911865,40.088440,91.084351,0.826001,5873.0,13772.0,3.0


In [11]:
#criando os crops das EMB identificadas no YOLO e que possuem correlação com AIS (both)
import PIL
from PIL import Image

#Resenting the maximum size
PIL.Image.MAX_IMAGE_PIXELS = None

# cortando as BoundBoxes nas imagens
# Opens a image in RGB mode
im = Image.open(filtered_files_png[-1])

# Cropped image of above dimension
# (It will not change original image)
#im = im.crop((left, top, right, bottom)) 

for i in range(len(df3.lat_pixel)):
    exec(f'im_{i} = im.crop((df3.lon_pixel.iloc[i]-320, df3.lat_pixel.iloc[i]-320, df3.lon_pixel.iloc[i]+320, df3.lat_pixel.iloc[i]+320))')
    exec(f'im_{i}.save(f"both{i+1}.png")')

        
# Shows the image in image viewer
#im0.show()

In [12]:
# Cria o dataframe com as EMB identificadas apenas pelo YOLO

only_yolo = df[~df['Latitude'].isin(df3['Latitude'])]



# cortando os crops na imagem dos alvos que não tinha correlação com AIS

for i in range(len(only_yolo.lat_pixel)):
    exec(f'im_{i} = im.crop((only_yolo.lon_pixel.iloc[i]-320, only_yolo.lat_pixel.iloc[i]-320, only_yolo.lon_pixel.iloc[i]+320, only_yolo.lat_pixel.iloc[i]+320))')
    exec(f'im_{i}.save("only_yolo{i+1}.png","PNG")')

# Shows the image in image viewer
#im0.show()


In [13]:

# Criando uma nova coluna chamada 'ImageFile' nos dataframes
files = os.listdir()

imagens_yolo = [file_  for file_ in files if file_.startswith('only_yolo')  & file_.endswith('.png')]
only_yolo['ImageFile']= imagens_yolo


imagens_both = [file_  for file_ in files if file_.startswith('both')  & file_.endswith('.png')]
df3['ImageFile']= sorted(imagens_both, key=len)



In [14]:
df3


,Latitude,Longitude,Vessel Name,x,y,width,height,score,lon_pixel,lat_pixel,Label,ImageFile
0,-13.581248,-36.787758,Amity,8026.569092,13071.831421,38.123291,53.208862,0.828042,8046.0,13098.0,1.0,both1.png
1,-13.378147,-36.938988,Mai Tai,10128.824280,11281.214966,30.093994,56.912964,0.827811,10144.0,11310.0,2.0,both2.png
2,-13.686136,-36.606274,Amity,5853.113525,13726.911865,40.088440,91.084351,0.826001,5873.0,13772.0,3.0,both3.png


In [15]:
largura_both = []
comprimento_both = []


for image in imagens_both:
    # Abra a imagem
    imagem = Image.open(image)

    # Converta a imagem para escala de cinza
    imagem_cinza = imagem.convert('L')

    # Obtenha a largura e a altura da imagem
    largura, altura = imagem_cinza.size

    # Acesse a matriz de intensidade dos pixels
    matriz_pixels = list(imagem_cinza.getdata())

    # Converta a matriz de intensidade dos pixels em uma matriz 2D
    matriz_pixels = [matriz_pixels[i * largura:(i + 1) * largura] for i in range(altura)]

    # Inicialize o tamanho do grupo de pixels
    tamanho_x = 0
    tamanho_y = 0

    # Verifique o grupo de pixels acima de 250 ao longo do eixo X (largura)
    x = largura // 2
    while x < largura and matriz_pixels[altura // 2][x] > 254:
        tamanho_x += 1
        x += 1

    # Verifique o grupo de pixels acima de 250 ao longo do eixo Y (altura)
    y = altura // 2
    while y < altura and matriz_pixels[y][largura // 2] > 254:
        tamanho_y += 1
        y += 1

    # Converta o tamanho do grupo de pixels em metros
    tamanho_real_x = tamanho_x * 10
    tamanho_real_y = tamanho_y * 10

#print(f'Quantidade de pixels em X: {tamanho_x}')
#print(f'Quantidade de pixels em Y: {tamanho_y}')
#print(f'Largura estimada (x): {tamanho_real_x} metros')
#print(f'Comprimento estimado (y): {tamanho_real_y} metros')
    largura_both.append(tamanho_real_x)
    comprimento_both.append(tamanho_real_y)

df3['Comprimento_aprox']= comprimento_both
df3['Largura_aprox']= largura_both


# Salva o dataframe em um arquivo csv
df3.to_csv('both.csv', index=False, sep='\t') 

In [16]:
largura_yolo = []
comprimento_yolo = []


for image in imagens_yolo:
    # Abra a imagem
    imagem = Image.open(image)

    # Converta a imagem para escala de cinza
    imagem_cinza = imagem.convert('L')

    # Obtenha a largura e a altura da imagem
    largura, altura = imagem_cinza.size

    # Acesse a matriz de intensidade dos pixels
    matriz_pixels = list(imagem_cinza.getdata())

    # Converta a matriz de intensidade dos pixels em uma matriz 2D
    matriz_pixels = [matriz_pixels[i * largura:(i + 1) * largura] for i in range(altura)]

    # Inicialize o tamanho do grupo de pixels
    tamanho_x = 0
    tamanho_y = 0

    # Verifique o grupo de pixels acima de 250 ao longo do eixo X (largura)
    x = largura // 2
    while x < largura and matriz_pixels[altura // 2][x] > 254:
        tamanho_x += 1
        x += 1

    # Verifique o grupo de pixels acima de 250 ao longo do eixo Y (altura)
    y = altura // 2
    while y < altura and matriz_pixels[y][largura // 2] > 254:
        tamanho_y += 1
        y += 1

    # Converta o tamanho do grupo de pixels em metros
    tamanho_real_x = tamanho_x * 10
    tamanho_real_y = tamanho_y * 10

#print(f'Quantidade de pixels em X: {tamanho_x}')
#print(f'Quantidade de pixels em Y: {tamanho_y}')
#print(f'Largura estimada (x): {tamanho_real_x} metros')
#print(f'Comprimento estimado (y): {tamanho_real_y} metros')
    largura_yolo.append(tamanho_real_x)
    comprimento_yolo.append(tamanho_real_y)

only_yolo['Comprimento_aprox']= comprimento_yolo
only_yolo['Largura_aprox']= largura_yolo


only_yolo.to_csv('only_yolo.csv', index=False, sep='\t')
only_yolo

,x,y,width,height,score,lon_pixel,lat_pixel,Latitude,Longitude,Label,ImageFile,Comprimento_aprox,Largura_aprox
3,13723.977295,11844.408508,14.363525,25.514160,0.663695,13731,11857,-13.352984,-37.272869,4,only_yolo1.png,70,20
4,13799.831787,12425.027893,20.251465,21.570679,0.649854,13810,12436,-13.403110,-37.292183,5,only_yolo2.png,40,10
5,23505.935547,13571.501953,398.107422,334.365234,0.623523,23705,13739,-13.312819,-38.208595,6,only_yolo3.png,0,0


In [17]:
# criando um mapa interativo com os resultados do YOLO
import folium
from folium.plugins import MeasureControl

# Criando um mapa centrado em uma localização inicial
mymap = folium.Map(location=[df3['Latitude'].mean(), df3['Longitude'].mean()], zoom_start=8)#, tiles="esri oceanbasemap")


# Coordenadas do polígono (AJ-2DN)
polygon_coords = [
           (-10.510072456616257,-36.40003390957367),
(-20.660801633731936,-10.000000661953115),
(-27.260000002416582,-9.999999161135598),
(-22.33969923144727,-26.343099096699575),
(-22.33776950713623,-26.341682890339698),
(-22.326043973516917,-26.33310613612875),
(-21.74385699226221,-26.009228105248486),
(-21.110223488733354,-25.81065103925653),
(-20.46058735311935,-25.763370785448917),
(-19.888222451617295,-25.81537906463729),
(-19.260241491541493,-26.0328682321523),
(-18.723885351253685,-26.30709370423644),
(-18.2013901610364,-26.737804856052065),
(-17.769260511846028,-27.256645205560265),
(-17.44874101961318,-27.84704974120753),
(-17.24826125912189,-28.47476104679674),
(-17.15626642227768,-29.134684398213782),
(-17.18171637612373,-29.80075610383657),
(-17.330430635819,-30.452481649645947),
(-17.5901820895576,-31.064058875009103),
(-17.951061322293498,-31.61213894595363),
(-18.405174758638054,-32.07826311843916),
(-18.94186697330315,-32.49917946346109),
(-19.264944499302544,-32.60440503990547),
(-19.529559537482058,-32.7183994143869),
(-20.36599862012173,-32.889390976109034),
(-18.31797390321944,-39.66620254783495),
(-18.23682102000906,-39.635976282182334),
(-18.134541379301695,-39.57930203408369),
(-17.75342716184666,-39.1864893085217),
(-17.17394266512416,-39.1782851972086),
(-16.626713304151522,-39.084462665606786),
(-16.455593885651638,-39.00156601823842),
(-15.849362519460158,-38.835772723501684),
(-15.56705393112999,-38.93142270123442),
(-14.649988118623511,-39.04620267451369),
(-13.698065278586782,-38.80858483562729),
(-13.15948664194903,-38.75757410957073),
(-13.018546090562737,-38.39017705469604),
(-12.3235309956641,-37.75558214173065),
(-11.113792361790928,-37.09872074059105),
(-10.821322903134202,-36.92610424503053),
(-10.6381422879832,-36.683956186135845),
(-10.510072456616257,-36.40003390957367),
(-10.510072456616257,-36.40003390957367)
       
]

# Adicionando o polígono ao mapa
folium.Polygon(locations=polygon_coords, color='blue', fill=True, fill_color='blue').add_to(mymap)

# Coordenadas do polígono (IMAGEM)
vertices = [
(lat_vertice_superior_esquerdo, long_vertice_superior_esquerdo), 
(lat_vertice_superior_direito, long_vertice_superior_direito),
(lat_vertice_inferior_direito, long_vertice_inferior_direito),
(lat_vertice_inferior_esquerdo, long_vertice_inferior_esquerdo)]

folium.Polygon(locations=vertices, color='gray', fill=True, fill_color='gray').add_to(mymap)

# Adicionando marcadores verde para cada ponto no DataFrame do both (df3)
for index, row in df3.iterrows():
    popup_content = f"""
        <p>Latitude: {row['Latitude']}</p>
        <p>Longitude: {row['Longitude']}</p>
        <p>Image: {row['ImageFile']}</p>
        <p>Width(m): {row['Largura_aprox']}</p>
        <p>Length(m): {row['Comprimento_aprox']}</p>
        <img src="{row['ImageFile']}" width="200">
    """
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=folium.Popup(html=popup_content, max_width=400),
        icon=folium.Icon(color="green")
    ).add_to(mymap)

    
# Adicionando marcadores vermelhos para os pontos do segundo DataFrame (only_yolo)
for index, row in only_yolo.iterrows():
    popup_content = f"""
        <p>Latitude: {row['Latitude']}</p>
        <p>Longitude: {row['Longitude']}</p>
        <p>Image: {row['ImageFile']}</p>
        <p>Width(m): {row['Largura_aprox']}</p>
        <p>Length(m): {row['Comprimento_aprox']}</p>
        <img src="{row['ImageFile']}" width="200">
    """
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=folium.Popup(html=popup_content, max_width=400),
        icon=folium.Icon(color='red', icon='fa-sharp fa-thin fa-circle-exclamation', prefix='fa')
    ).add_to(mymap)
    
# Adicionando marcadores verde para cada ponto no DataFrame do both (df3)
for index, row in da.iterrows():
    popup_content = f"""
        <p>Fonte AIS: Seavision</p>
        <p>Nome Navio: {row['Vessel Name']}</p>
        <p>Veloc: {row['Speed']} nós</p>
        <p>Rumo: {row['Heading']}º</p>
        <p>Comprimento: {row['Length']} m</p>
        <p>Largura: {row['Beam']} m</p>
        #<p>Idade AIS: {row['Minutos_Tempo']} min</p>
        <p>Raio Alcance: {row['Raio_alcance']} MN</p>
    """
    folium.Circle(
        radius=600,
        location=[row['Latitude'], row['Longitude']],
        popup=folium.Popup(html=popup_content, max_width=400),
        color='blue', fill=True
    ).add_to(mymap)

# Criando uma barra lateral (sidebar) com opções de seleção
#folium.LayerControl().add_to(mymap)
mymap.add_child(MeasureControl())
# Salvando o mapa como um arquivo HTML
mymap.save('mapa_interativo.html')

#import io
#from PIL import Image

#img_data = mymap._to_png(5)
#img = Image.open(io.BytesIO(img_data))
#img.save('mapa.png')

print("Mapa interativo criado com sucesso! Verifique o arquivo 'mapa_interativo.html'.")
mymap

Mapa interativo criado com sucesso! Verifique o arquivo 'mapa_interativo.html'.


In [18]:
# filtrando os arquivos a serem enviados via email
files
filtered = [file_  for file_ in files if file_.endswith('.nc') ]
zipp = filtered[-1][63:67]
zipp = zipp +'.zip'

files = os.listdir()
filtered = [file_  for file_ in files if file_.startswith('only_yolo')  & file_.endswith('.png') | file_.startswith('both') & file_.endswith('.png') ]
zipp


'2C90.zip'

# Abaixo desse linha você terá os resultados dessa inferência. 

Os arquivos brutos poderão ser acessados individualmente 

In [19]:

# Todos os alvos
print('Estes foram os alvos identificados pela Rede')
print(f'Foram identificados {len(df.Label)} alvos.')
df

Estes foram os alvos identificados pela Rede
Foram identificados 6 alvos.


,x,y,width,height,score,lon_pixel,lat_pixel,Latitude,Longitude,Label
0,8026.569092,13071.831421,38.123291,53.208862,0.828042,8046,13098,-13.581248,-36.787758,1
1,10128.824280,11281.214966,30.093994,56.912964,0.827811,10144,11310,-13.378147,-36.938988,2
2,5853.113525,13726.911865,40.088440,91.084351,0.826001,5873,13772,-13.686136,-36.606274,3
3,13723.977295,11844.408508,14.363525,25.514160,0.663695,13731,11857,-13.352984,-37.272869,4
4,13799.831787,12425.027893,20.251465,21.570679,0.649854,13810,12436,-13.403110,-37.292183,5
5,23505.935547,13571.501953,398.107422,334.365234,0.623523,23705,13739,-13.312819,-38.208595,6


In [20]:
# Alvos com correlaçao AIS

print('Abaixo estão os alvos identificados pela Rede que possuem correlação com alguma fonte AIS')
print('  ')
print(f'Do total de {len(df.Label)} alvos identificados pela Rede, {len(df3.Label)} possuem correlação com uma fonte AIS.')
print('Por favor, verifique nas imagens anexas a veracidade da identificação dos alvos pela rede. Erros devem ser reportados ao administrador visando a melhoria do sistema')
print('  ')
print('Os dados AIS utilizados possuem até uma hora diferença em relação ao momento da captura da imagem satelital. ')
print('Desta forma, foi necessário utilizar um raio de +- 50km para correlacionar um dado AIS com um alvo da rede.')
print('Com isso, alguns alvos YOLO podem se correlacionar com mais de um navio ou um navio pode se correlacionar com mais de um alvo YOLO.')
print('  ')
df2



Abaixo estão os alvos identificados pela Rede que possuem correlação com alguma fonte AIS
  
Do total de 6 alvos identificados pela Rede, 3 possuem correlação com uma fonte AIS.
Por favor, verifique nas imagens anexas a veracidade da identificação dos alvos pela rede. Erros devem ser reportados ao administrador visando a melhoria do sistema
  
Os dados AIS utilizados possuem até uma hora diferença em relação ao momento da captura da imagem satelital. 
Desta forma, foi necessário utilizar um raio de +- 50km para correlacionar um dado AIS com um alvo da rede.
Com isso, alguns alvos YOLO podem se correlacionar com mais de um navio ou um navio pode se correlacionar com mais de um alvo YOLO.
  


,Latitude,Longitude,Vessel Name,x,y,width,height,score,lon_pixel,lat_pixel,Label
0,-13.581248,-36.787758,Amity,8026.569092,13071.831421,38.123291,53.208862,0.828042,8046.0,13098.0,1.0
1,-13.378147,-36.938988,Mai Tai,10128.824280,11281.214966,30.093994,56.912964,0.827811,10144.0,11310.0,2.0
2,-13.686136,-36.606274,Amity,5853.113525,13726.911865,40.088440,91.084351,0.826001,5873.0,13772.0,3.0
3,-13.686136,-36.606274,Nc Brisa,5853.113525,13726.911865,40.088440,91.084351,0.826001,5873.0,13772.0,3.0


In [21]:



# !!!!! POSSÍVEIS EMB NÃO COLABORATIVAS!!! Alvos sem correlação AIS
print('ATENÇÃO!!!Possível EMB não colaborativa!!!')

print('  ')
print('Abaixo estão os alvos identificados pela Rede que NÃO possuem correlação com alguma fonte AIS')
print('  ')
print(f'Do total de {len(df.Label)} alvos identificados pela Rede, {len(only_yolo.Label)} não possuem correlação com uma fonte AIS.')
print('Por favor, verifique nas imagens anexas a veracidade da identificação dos alvos pela rede.')
print(' Erros devem ser reportados ao administrador')
only_yolo

ATENÇÃO!!!Possível EMB não colaborativa!!!
  
Abaixo estão os alvos identificados pela Rede que NÃO possuem correlação com alguma fonte AIS
  
Do total de 6 alvos identificados pela Rede, 3 não possuem correlação com uma fonte AIS.
Por favor, verifique nas imagens anexas a veracidade da identificação dos alvos pela rede.
 Erros devem ser reportados ao administrador


,x,y,width,height,score,lon_pixel,lat_pixel,Latitude,Longitude,Label,ImageFile,Comprimento_aprox,Largura_aprox
3,13723.977295,11844.408508,14.363525,25.514160,0.663695,13731,11857,-13.352984,-37.272869,4,only_yolo1.png,70,20
4,13799.831787,12425.027893,20.251465,21.570679,0.649854,13810,12436,-13.403110,-37.292183,5,only_yolo2.png,40,10
5,23505.935547,13571.501953,398.107422,334.365234,0.623523,23705,13739,-13.312819,-38.208595,6,only_yolo3.png,0,0


In [22]:
# mapa interativo

print('Mapa com os alvos colaborativos (verde) e não colaborativos (vermelho). ')
mymap

Mapa com os alvos colaborativos (verde) e não colaborativos (vermelho). 


 # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #

In [23]:

#convertendo esse arquivo ipynb em html
os.system('jupyter nbconvert --to html sahi.ipynb')

import zipfile
with zipfile.ZipFile(zipp, 'w') as zip:
    # Adicione os arquivos que deseja compactar ao arquivo ZIP
    zip.write('both.csv')
    zip.write('only_yolo.csv')
    zip.write('seavision.csv')
    zip.write('mapa_interativo.html')
    zip.write('sahi.html')
    for i in range(len(filtered)):
        zip.write(filtered[i])
        
cur_dir = os.getcwd()

In [24]:

# criar um email automático com os resultados

import win32com.client as win32

# criar a integração com o outlook
outlook = win32.Dispatch('outlook.application')

ano = filtered_files_png[-1][19:21]
mes = filtered_files_png[-1][21:23]
dia = filtered_files_png[-1][23:25]

email = outlook.CreateItem(0)

imagem = (filtered_files_png[-1])

# configurar as informações do seu e-mail
email.To = "camilla.caricchio@hotmail.com; wellington.sa@marinha.mil.br" #"destino; destino2"
email.Subject = "E-mail automático com os resultados ENC"
email.HTMLBody = f"""
<p>Prezado Operador, </p>

</p> Segue o relatório da análise da imagem Satelital {imagem}, coletada em {dia}/{mes}/{ano}. </p>

<p> Dentro do arquivo zip anexo a este e-mail estão:</p>

<p> - Arquivo sahi.html contendo o relatório da inferência. <p>
<p> - Mapa interativo com as posições dos alvos encontrados. Alvos com correlação AIS em verde e sem correlação (possível EMB não colaborativa) em vermelho</p>
<p> - Arquivo "only_yolo.csv', contendo a Relação de embarcações encontradas somente no Yolo e os respectivos crops das imagens</p>
<p> - Arquivo "both.csv', contendo a  Relação de embarcações encontradas no Yolo com correlação com dados AIS e os respectivos crops das imagens</p>
<p> - Arquivo "seavision.csv', contendo a  Relação de embarcações OBS no Seavision na mesma área e horário da coleta da imagem satelital</p>

<p>Atenciosamente,</p>

"""
anexo = cur_dir + '\\' +zipp
email.Attachments.Add(anexo)

email.Send()